# Notebook 06: Visual Synthesis & Final Experimental Report Compilation
## Phase 6 of the Experimental Workflow (`results/final_experimental_report.md`)

**Scientific & Publication Deliverables:**
1. **T6.1: Dual-Polarity $10 \times 6$ Spatio-Temporal Heatmaps (+ Red / - Blue)**:
   - High-resolution (300 DPI) IEEE heatmaps illustrating normalized signed Shapley attributions.
2. **T6.2: Comparative MoRF vs. LeRF Curves Across All 3 Models**:
   - Unified degradation plots highlighting causal faithfulness gaps (AUDC) and step $k=1$ single-field ablation behavior.
3. **T6.3: Automated Final Experimental Summary Report Compilation**:
   - Generates `results/final_experimental_report.md` consolidating benchmark classification, edge latency, XAI audit, and causal perturbation tables.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
RESULTS_DIR = REPO_ROOT / 'results'
METRICS_DIR = RESULTS_DIR / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'

for d in [METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("[OK] Setup complete for visual synthesis and final reporting.")

### 6.1 Dual-Polarity $10 \times 6$ Heatmap Generation (+ Red / - Blue) (Task T6.1)
Render instance-normalized signed SHAP attribution maps with a diverging colormap (`coolwarm`), where:
- **Red (+)** indicates an excitatory contribution supporting the predicted class;
- **Blue (-)** indicates an inhibitory contribution opposing the predicted class.

In [ ]:
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
npz_path = METRICS_DIR / 'xai_explanations.npz'

npz_data = np.load(npz_path, allow_pickle=True)  # trusted artifact generated by Notebook 03
hm_lopez = np.median(npz_data['shap_lopez_cell'], axis=0)    # [10, 6]
hm_biflow = np.median(npz_data['shap_biflow_cell'], axis=0)  # [10, 6]
hm_meddef = np.median(npz_data['shap_meddef_cell'], axis=0)  # [10, 6]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
v_limit = max(np.abs(hm_lopez).max(), np.abs(hm_biflow).max(), np.abs(hm_meddef).max())

heatmaps = [('Lopez17CNN (Black-Box)', hm_lopez), 
             ('BiflowTransformer (Sequential)', hm_biflow), 
             ('MedDefenderTransformer (By-Design)', hm_meddef)]

for idx, (title, hm) in enumerate(heatmaps):
    ax = axes[idx]
    sns.heatmap(
        hm,
        ax=ax,
        cmap='coolwarm',
        center=0,
        vmin=-v_limit,
        vmax=v_limit,
        xticklabels=FIELDS,
        yticklabels=[f'Pkt {t+1}' for t in range(10)],
        cbar=(idx == 2),
        cbar_ax=cbar_ax if (idx == 2) else None,
        linewidths=0.5,
        linecolor='white'
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(r'Protocol Header Fields, $f \in \{PL, IAT, DIR, WIN, TTL, FLG\}$', fontsize=10)
    if idx == 0:
        ax.set_ylabel(r'Packet Sequence Index, $t \in [1..10]$', fontsize=10)
cbar_ax.set_ylabel(r'Instance-Normalized Signed SHAP $\tilde{\phi}_{t,f}$ (+ Red / - Blue) [a.u.]', fontsize=10)

plt.suptitle('Spatio-Temporal XAI Heatmaps (+ Red Excitatory / - Blue Inhibitory) - Eq. 5 Nascita et al.', fontsize=13, y=1.02)
out_hm_png = FIGURES_DIR / 'xai_heatmaps_multimodel.png'
out_hm_pdf = FIGURES_DIR / 'xai_heatmaps_multimodel.pdf'
plt.savefig(out_hm_png, dpi=300, bbox_inches='tight')
plt.savefig(out_hm_pdf, bbox_inches='tight')
plt.show()
print(f"[OK] Heatmaps saved to:\n  - {out_hm_png}\n  - {out_hm_pdf}")

### 6.2 Comparative MoRF vs. LeRF Degradation Plots (Task T6.2)
Visualize progressive inference-time accuracy degradation ($m=0..5$ header fields neutralized at `0.5` on frozen models).

In [ ]:
# Degradation curves measured in Notebook 05
df_curves = pd.read_csv(METRICS_DIR / 'morf_lerf_curves.csv')
steps = [0, 1, 2, 3, 4, 5]
step_cols = [f'm={i}' for i in steps]
morf_data = {r['Model']: [r[c] for c in step_cols] for _, r in df_curves[df_curves['Curve'] == 'MoRF_F1'].iterrows()}
lerf_data = {r['Model']: [r[c] for c in step_cols] for _, r in df_curves[df_curves['Curve'] == 'LeRF_F1'].iterrows()}

plt.figure(figsize=(10, 6))
colors = {'Lopez17CNN': '#d95f02', 'BiflowTransformer': '#7570b3', 'MedDefenderTransformer': '#1b9e77'}

for net in morf_data.keys():
    plt.plot(steps, morf_data[net], marker='o', linewidth=2.2, color=colors[net], label=f"{net} (MoRF - Most Relevant)")
    plt.plot(steps, lerf_data[net], marker='s', linestyle='--', linewidth=1.8, color=colors[net], alpha=0.7, label=f"{net} (LeRF - Least Relevant)")

plt.axvline(x=1, color='gray', linestyle=':', linewidth=1.5, label='Step m=1 (top field)')
plt.title('Inference-Time Causal Validation: MoRF vs. LeRF Curves on Frozen Models (RQ3)', fontsize=12, fontweight='bold')
plt.xlabel(r'Number of Neutralized Protocol Fields at Inference, $m \in [0..5]$', fontsize=11)
plt.ylabel('Macro F1-Score [%]', fontsize=11)
plt.xticks(steps, ['m=0\n(Intact)', 'm=1\n(top field)', 'm=2', 'm=3', 'm=4', 'm=5'])
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower left', fontsize=9)

out_curves_png = FIGURES_DIR / 'morf_lerf_curves.png'
out_curves_pdf = FIGURES_DIR / 'morf_lerf_curves.pdf'
out_morf_vs_lerf_png = FIGURES_DIR / 'morf_vs_lerf_curves.png'
out_morf_vs_lerf_pdf = FIGURES_DIR / 'morf_vs_lerf_curves.pdf'

plt.savefig(out_curves_png, dpi=300, bbox_inches='tight')
plt.savefig(out_curves_pdf, bbox_inches='tight')
plt.savefig(out_morf_vs_lerf_png, dpi=300, bbox_inches='tight')
plt.savefig(out_morf_vs_lerf_pdf, bbox_inches='tight')
plt.show()
print(f"[OK] MoRF vs. LeRF figures saved to:\n  - {out_curves_png}\n  - {out_morf_vs_lerf_png}")

### 6.3 By-Design Inter-Packet Attention Matrices ($10 \times 10$) (Task T6.1b)
Plot the inter-packet self-attention matrices ($10 \times 10$) extracted during the forward pass at zero extra cost:
- **BiflowTransformer**: joint inter-packet *Attention Rollout* matrix across flow packets;
- **MedDefenderTransformer**: *MHAN* pooling self-attention weights across the 10 packets.
All axes and colorbars specify explicit units.

In [ ]:
# Flow-averaged 10x10 inter-packet matrices on explained flows (extracted in Notebook 03)
T = 10
mat_biflow = npz_data['attn_biflow_mat']
mat_biflow = mat_biflow / mat_biflow.sum(axis=1, keepdims=True)  # submatrix without CLS: rows renormalized to sum to 1
mat_meddef = npz_data['attn_meddef_mat']
v_max = max(mat_biflow.max(), mat_meddef.max())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cbar_ax2 = fig.add_axes([0.92, 0.15, 0.015, 0.7])

# BiflowTransformer Rollout
sns.heatmap(
    mat_biflow, ax=axes[0], cmap='Blues', vmin=0, vmax=v_max,
    xticklabels=[f'P{j+1}' for j in range(T)], yticklabels=[f'P{i+1}' for i in range(T)],
    cbar=False, square=True, linewidths=0.5, linecolor='white'
)
axes[0].set_title('BiflowTransformer: Attention Rollout Matrix (By-Design)', fontweight='bold', fontsize=11)
axes[0].set_xlabel(r'Key Packet Index, $j \in [1..10]$', fontsize=10)
axes[0].set_ylabel(r'Query Packet Index, $i \in [1..10]$', fontsize=10)

# MedDefender MHAN
sns.heatmap(
    mat_meddef, ax=axes[1], cmap='Greens', vmin=0, vmax=v_max,
    xticklabels=[f'P{j+1}' for j in range(T)], yticklabels=[f'P{i+1}' for i in range(T)],
    cbar=True, cbar_ax=cbar_ax2, square=True, linewidths=0.5, linecolor='white'
)
axes[1].set_title('MedDefenderTransformer: MHAN Pooling Weights (By-Design)', fontweight='bold', fontsize=11)
axes[1].set_xlabel(r'Key Packet Index, $j \in [1..10]$', fontsize=10)
axes[1].set_ylabel(r'Query Packet Index, $i \in [1..10]$', fontsize=10)
cbar_ax2.set_ylabel(r'Inter-Packet Attention Weight $\alpha_{i,j}$ [Dimensionless share, $\Sigma=1.0$]', fontsize=10)

plt.suptitle(r'Inter-Packet Self-Attention Matrices ($10 \times 10$) - Zero-Cost Intrinsic Explainability', fontsize=13, y=1.02)
out_attn_png = FIGURES_DIR / 'attention_matrices_inter_packet.png'
out_attn_pdf = FIGURES_DIR / 'attention_matrices_inter_packet.pdf'
plt.savefig(out_attn_png, dpi=300, bbox_inches='tight')
plt.savefig(out_attn_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Attention matrices saved to:\n  - {out_attn_png}\n  - {out_attn_pdf}')

### 6.4 Statistical Distribution of Header Field Importance ($F=6$) (Task T6.1c)
Visualize the statistical spread of instance-normalized absolute SHAP attribution $|\tilde{\phi}_f|$ across test flows for each protocol header field (`PL`, `IAT`, `DIR`, `WIN`, `TTL`, `FLG`).
Axes and legend include **explicit measurement units**.

In [ ]:
# Collect per-sample distributions
npz_path = METRICS_DIR / 'xai_explanations.npz'
npz_data = np.load(npz_path, allow_pickle=True)  # trusted artifact generated by Notebook 03
f_lopez = np.abs(npz_data['shap_lopez_field'])
f_biflow = np.abs(npz_data['shap_biflow_field'])
f_meddef = np.abs(npz_data['shap_meddef_field'])

records_df = []
for i in range(len(f_lopez)):
    for j, fld in enumerate(FIELDS):
        records_df.append({'Field': fld, 'Model': 'Lopez17CNN', 'Attribution': f_lopez[i, j]})
        records_df.append({'Field': fld, 'Model': 'BiflowTransformer', 'Attribution': f_biflow[i, j]})
        records_df.append({'Field': fld, 'Model': 'MedDefenderTransformer', 'Attribution': f_meddef[i, j]})

df_box = pd.DataFrame(records_df)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_box, x='Field', y='Attribution', hue='Model', palette={'Lopez17CNN': '#d95f02', 'BiflowTransformer': '#7570b3', 'MedDefenderTransformer': '#1b9e77'})
plt.title('Per-Flow Header Field Importance Distribution on Test Flows (Eq. 5 Nascita et al.)', fontweight='bold', fontsize=12)
plt.xlabel(r'Protocol Header Fields, $f \in \{PL, IAT, DIR, WIN, TTL, FLG\}$', fontsize=11)
plt.ylabel(r'Normalized Absolute Attribution per Flow $|\tilde{\phi}_f|$ [Dimensionless share (a.u.)]', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5, axis='y')
plt.legend(title='Network Architecture', loc='upper right')

out_box_png = FIGURES_DIR / 'field_attribution_distributions_boxplot.png'
out_box_pdf = FIGURES_DIR / 'field_attribution_distributions_boxplot.pdf'
plt.savefig(out_box_png, dpi=300, bbox_inches='tight')
plt.savefig(out_box_pdf, bbox_inches='tight')
plt.show()
print(f'[SUCCESS] Distribution boxplot saved to:\n  - {out_box_png}\n  - {out_box_pdf}')

### 6.5 Automated Experimental Summary Report Compilation (Task T6.3)
Load CSV outputs from Phases 2, 4, and 5 and compile the Markdown summary into `results/final_experimental_report.md`.

In [ ]:
# Tables generated by Notebooks 02, 04, and 05
df_bench = pd.read_csv(METRICS_DIR / 'benchmark_metrics.csv')
df_audit = pd.read_csv(METRICS_DIR / 'xai_audit_table.csv')
df_mask = pd.read_csv(METRICS_DIR / 'masking_results.csv')

report_md = rf"""**Project**: Explainable AI (XAI) for Early Intrusion Detection in IoMT Networks (DIETI — University of Naples Federico II)  
**Dataset**: CICIoMT2024 (13-class multiclass traffic, $T=10$ packets, $F=6$ header fields, `seed=0`)  

---

## 1. Benchmark Classification Performance & Single-Thread Edge CPU Efficiency
{df_bench.to_markdown(index=False)}

---

## 2. Formal XAI Audit: By-Design Attention vs. Post-Hoc GradientSHAP
{df_audit.to_markdown(index=False)}

---

## 3. Causal Validation on Frozen Models (MoRF vs. LeRF Perturbation)
{df_mask.to_markdown(index=False)}

"""

out_report_file = RESULTS_DIR / 'final_experimental_report.md'
with open(out_report_file, 'w', encoding='utf-8') as f:
    f.write(report_md)

print(f"\n[SUCCESS] Summary report compiled successfully to:\n  {out_report_file}")